<h1 style="color:DodgerBlue">Индивидальный проект</h1>

<h2 style="color:DodgerBlue">Название проекта:</h2>

----

### Вариант задания 17 


<h2 style="color:DodgerBlue">Описание проекта:</h2>

----

Создать  базовый  класс ShippingOption в  C#,  который  будет  представлять 
различные опции доставки. На основе этого класса разработать 2-3 производных 
класса, демонстрирующих принципы наследования и полиморфизма. В каждом из 
классов  должны  быть  реализованы  новые  атрибуты  и  методы,  а  также 
переопределены  некоторые  методы  базового  класса  для  демонстрации 
полиморфизма.

#### Дополнительное задание
Добавьте к сущестующим классам (базовыму и производным 3-4 атрибута и метода) создайте явную реализации интерфейса и управление зависимостями 


<h2 style="color:DodgerBlue">Реализация:</h2>

----

In [4]:
using System;
using System.Collections.Generic;

public interface INotificationService
{
    void SendNotification(string message);
    void SendNotification(string message, string recipient);
    bool CanSendNotification();
}

public interface IDiscountService
{
    decimal CalculateDiscount(decimal baseCost);
    decimal CalculateDiscount(decimal baseCost, string promoCode);
    bool IsDiscountAvailable();
}

public class EmailNotificationService : INotificationService
{
    private string _smtpServer;
    private int _port;
    private bool _isConfigured;

    public string SmtpServer 
    { 
        get => _smtpServer;
        set => _smtpServer = !string.IsNullOrEmpty(value) ? value : "smtp.default.com";
    }

    public int Port 
    { 
        get => _port;
        set => _port = value > 0 ? value : 587;
    }

    public bool IsConfigured 
    { 
        get => _isConfigured;
        set => _isConfigured = value;
    }

    public EmailNotificationService(string smtpServer = "smtp.default.com", int port = 587)
    {
        SmtpServer = smtpServer;
        Port = port;
        IsConfigured = !string.IsNullOrEmpty(smtpServer);
    }

    void INotificationService.SendNotification(string message)
    {
        if (((INotificationService)this).CanSendNotification())
        {
            Console.WriteLine($"[EMAIL] Отправка уведомления: {message}");
            Console.WriteLine($"Через сервер: {SmtpServer}:{Port}");
        }
        else
        {
            Console.WriteLine("[EMAIL] Невозможно отправить уведомление - сервис не настроен");
        }
    }

    void INotificationService.SendNotification(string message, string recipient)
    {
        if (((INotificationService)this).CanSendNotification())
        {
            Console.WriteLine($"[EMAIL] Отправка уведомления для {recipient}: {message}");
            Console.WriteLine($"Через сервер: {SmtpServer}:{Port}");
        }
        else
        {
            Console.WriteLine("[EMAIL] Невозможно отправить уведомление - сервис не настроен");
        }
    }

    bool INotificationService.CanSendNotification()
    {
        return IsConfigured && !string.IsNullOrEmpty(SmtpServer);
    }

    public void SetConfiguration(string smtpServer, int port)
    {
        SmtpServer = smtpServer;
        Port = port;
        IsConfigured = true;
        Console.WriteLine($"Конфигурация email обновлена: {smtpServer}:{port}");
    }

    public string GetConfiguration()
    {
        return $"SMTP: {SmtpServer}:{Port}, Настроен: {IsConfigured}";
    }

    public bool ValidateConfiguration()
    {
        return IsConfigured && Port > 0 && !string.IsNullOrEmpty(SmtpServer);
    }
}

public class SeasonalDiscountService : IDiscountService
{
    private decimal _seasonalDiscount;
    private DateTime _seasonStart;
    private DateTime _seasonEnd;

    public decimal SeasonalDiscount 
    { 
        get => _seasonalDiscount;
        set => _seasonalDiscount = value >= 0 && value <= 0.5m ? value : 0.1m;
    }

    public DateTime SeasonStart 
    { 
        get => _seasonStart;
        set => _seasonStart = value;
    }

    public DateTime SeasonEnd 
    { 
        get => _seasonEnd;
        set => _seasonEnd = value;
    }

    public SeasonalDiscountService(decimal discount = 0.1m)
    {
        SeasonalDiscount = discount;
        SeasonStart = new DateTime(DateTime.Now.Year, 1, 1);
        SeasonEnd = new DateTime(DateTime.Now.Year, 12, 31);
    }

    decimal IDiscountService.CalculateDiscount(decimal baseCost)
    {
        if (((IDiscountService)this).IsDiscountAvailable())
        {
            decimal discount = baseCost * SeasonalDiscount;
            Console.WriteLine($"Применена сезонная скидка: {SeasonalDiscount:P0} = {discount} руб.");
            return discount;
        }
        return 0;
    }

    decimal IDiscountService.CalculateDiscount(decimal baseCost, string promoCode)
    {
        decimal discount = ((IDiscountService)this).CalculateDiscount(baseCost);
        
        if (!string.IsNullOrEmpty(promoCode) && promoCode.ToUpper() == "SUMMER2024")
        {
            decimal promoDiscount = baseCost * 0.05m;
            discount += promoDiscount;
            Console.WriteLine($"Дополнительная скидка по промокоду: 5% = {promoDiscount} руб.");
        }
        
        return discount;
    }

    bool IDiscountService.IsDiscountAvailable()
    {
        DateTime now = DateTime.Now;
        return now >= SeasonStart && now <= SeasonEnd;
    }

    public void SetSeason(DateTime start, DateTime end)
    {
        if (start < end)
        {
            SeasonStart = start;
            SeasonEnd = end;
            Console.WriteLine($"Сезон скидок установлен: {start:dd.MM.yyyy} - {end:dd.MM.yyyy}");
        }
        else
        {
            Console.WriteLine("Ошибка: дата начала должна быть раньше даты окончания");
        }
    }

    public bool IsSeasonActive()
    {
        return ((IDiscountService)this).IsDiscountAvailable();
    }

    public int DaysUntilSeasonEnd()
    {
        if (DateTime.Now < SeasonEnd)
        {
            return (SeasonEnd - DateTime.Now).Days;
        }
        return 0;
    }
}

public class DependencyContainer
{
    private readonly Dictionary<Type, object> _services;

    public DependencyContainer()
    {
        _services = new Dictionary<Type, object>();
    }

    public void Register<T>(T service)
    {
        _services[typeof(T)] = service;
    }

    public T GetService<T>()
    {
        if (_services.ContainsKey(typeof(T)))
        {
            return (T)_services[typeof(T)];
        }
        throw new InvalidOperationException($"Сервис {typeof(T).Name} не зарегистрирован");
    }

    public bool IsRegistered<T>()
    {
        return _services.ContainsKey(typeof(T));
    }
}

public class ShippingOption
{
    private string _deliveryOptionId;
    private string _deliveryOptionName;
    private decimal _cost;
    private DateTime _createdDate;
    private bool _isActive;
    private int _priorityLevel;
    private string _providerName;
    private int _maxDeliveryAttempts;
    private bool _requiresSignature;

    public string DeliveryOptionId 
    { 
        get => _deliveryOptionId;
        set => _deliveryOptionId = !string.IsNullOrEmpty(value) ? value : throw new ArgumentException("ID не может быть пустым");
    }
    
    public string DeliveryOptionName 
    { 
        get => _deliveryOptionName;
        set => _deliveryOptionName = !string.IsNullOrEmpty(value) ? value : throw new ArgumentException("Название не может быть пустым");
    }
    
    public decimal Cost 
    { 
        get => _cost;
        set => _cost = value >= 0 ? value : throw new ArgumentException("Стоимость не может быть отрицательной");
    }

    public DateTime CreatedDate 
    { 
        get => _createdDate;
        set => _createdDate = value;
    }

    public bool IsActive 
    { 
        get => _isActive;
        set => _isActive = value;
    }

    public int PriorityLevel 
    { 
        get => _priorityLevel;
        set => _priorityLevel = value >= 1 && value <= 5 ? value : throw new ArgumentException("Приоритет должен быть от 1 до 5");
    }

    public string ProviderName 
    { 
        get => _providerName;
        set => _providerName = !string.IsNullOrEmpty(value) ? value : "Неизвестный провайдер";
    }

    public int MaxDeliveryAttempts 
    { 
        get => _maxDeliveryAttempts;
        set => _maxDeliveryAttempts = value >= 1 && value <= 5 ? value : 3;
    }

    public bool RequiresSignature 
    { 
        get => _requiresSignature;
        set => _requiresSignature = value;
    }

    protected readonly INotificationService _notificationService;
    protected readonly IDiscountService _discountService;

    public ShippingOption(string id, string name, decimal cost, 
                         INotificationService notificationService = null, 
                         IDiscountService discountService = null)
    {
        DeliveryOptionId = id;
        DeliveryOptionName = name;
        Cost = cost;
        CreatedDate = DateTime.Now;
        IsActive = true;
        PriorityLevel = 3;
        ProviderName = "Standard Provider";
        MaxDeliveryAttempts = 3;
        RequiresSignature = false;
        _notificationService = notificationService;
        _discountService = discountService;
    }

    public virtual void SendCreationNotification()
    {
        _notificationService?.SendNotification($"Создана новая опция доставки: {DeliveryOptionName}");
    }

    public virtual decimal CalculateFinalCost()
    {
        decimal baseCost = CalculateCost();
        decimal discount = _discountService?.CalculateDiscount(baseCost) ?? 0;
        return baseCost - discount;
    }

    public virtual string GetProviderInfo()
    {
        return $"Провайдер: {ProviderName}, Макс. попыток доставки: {MaxDeliveryAttempts}, Требуется подпись: {RequiresSignature}";
    }

    public virtual void UpdateProvider(string providerName, int maxAttempts, bool requiresSignature)
    {
        ProviderName = providerName;
        MaxDeliveryAttempts = maxAttempts;
        RequiresSignature = requiresSignature;
        Console.WriteLine($"Информация о провайдере обновлена: {GetProviderInfo()}");
    }

    public virtual bool ValidateConfiguration()
    {
        return !string.IsNullOrEmpty(DeliveryOptionId) && 
               !string.IsNullOrEmpty(DeliveryOptionName) && 
               Cost >= 0 && 
               !string.IsNullOrEmpty(ProviderName) &&
               MaxDeliveryAttempts >= 1;
    }

    public virtual string GetSecurityInfo()
    {
        return $"Требуется подпись: {RequiresSignature}, Попытки доставки: {MaxDeliveryAttempts}, Приоритет: {PriorityLevel}";
    }
    
    public virtual decimal CalculateCost() => Cost;
    
    public virtual string EstimateDeliveryTime() => "Время доставки не указано";
    
    public virtual string GetDeliveryDetails() => 
        $"ID: {DeliveryOptionId}, Название: {DeliveryOptionName}, Стоимость: {Cost} руб.";

    public virtual string GetDeliveryDetails(bool includeStatus)
    {
        var details = GetDeliveryDetails();
        if (includeStatus)
        {
            details += $", Статус: {GetStatus()}";
        }
        return details;
    }

    public virtual string GetDeliveryDetails(bool includeStatus, bool includePriority)
    {
        var details = GetDeliveryDetails(includeStatus);
        if (includePriority)
        {
            details += $", Приоритет: {PriorityLevel}";
        }
        return details;
    }

    public virtual void UpdateCost(decimal newCost)
    {
        Cost = newCost;
        Console.WriteLine($"Стоимость обновлена: {newCost} руб.");
    }

    public virtual void UpdateCost(decimal newCost, string reason)
    {
        Cost = newCost;
        Console.WriteLine($"Стоимость обновлена: {newCost} руб. Причина: {reason}");
    }

    public virtual void ActivateOption()
    {
        IsActive = true;
        Console.WriteLine($"Опция {DeliveryOptionName} активирована");
    }

    public virtual void DeactivateOption()
    {
        IsActive = false;
        Console.WriteLine($"Опция {DeliveryOptionName} деактивирована");
    }

    public virtual string GetStatus()
    {
        return IsActive ? "Активна" : "Неактивна";
    }

    public virtual bool CanCombineWith(ShippingOption other) => false;
}

public class StandardDelivery : ShippingOption
{
    private int _averageDeliveryTime;
    private int _maxWeight;
    private string _deliveryRegion;
    private bool _hasTracking;
    private string _packageType;
    private decimal _insuranceCost;

    public int AverageDeliveryTime 
    { 
        get => _averageDeliveryTime;
        set => _averageDeliveryTime = value > 0 ? value : throw new ArgumentException("Время доставки должно быть положительным");
    }

    public int MaxWeight 
    { 
        get => _maxWeight;
        set => _maxWeight = value > 0 ? value : throw new ArgumentException("Вес должен быть положительным");
    }

    public string DeliveryRegion 
    { 
        get => _deliveryRegion;
        set => _deliveryRegion = !string.IsNullOrEmpty(value) ? value : "Все регионы";
    }

    public bool HasTracking 
    { 
        get => _hasTracking;
        set => _hasTracking = value;
    }

    public string PackageType 
    { 
        get => _packageType;
        set => _packageType = !string.IsNullOrEmpty(value) ? value : "Стандартная";
    }

    public decimal InsuranceCost 
    { 
        get => _insuranceCost;
        set => _insuranceCost = value >= 0 ? value : 0;
    }

    public StandardDelivery(string id, string name, decimal cost, int avgTime, 
                           INotificationService notificationService = null, 
                           IDiscountService discountService = null,
                           int maxWeight = 50, string region = "Все регионы") 
        : base(id, name, cost, notificationService, discountService)
    {
        AverageDeliveryTime = avgTime;
        MaxWeight = maxWeight;
        DeliveryRegion = region;
        HasTracking = true;
        PackageType = "Стандартная";
        InsuranceCost = cost * 0.01m;
    }

    public override decimal CalculateFinalCost()
    {
        decimal baseCost = CalculateCost();
        decimal discount = _discountService?.CalculateDiscount(baseCost) ?? 0;
        decimal total = baseCost - discount + InsuranceCost;
        
        Console.WriteLine($"Итоговая стоимость: {baseCost} - {discount} + {InsuranceCost} (страховка) = {total} руб.");
        return total;
    }

    public string GetTrackingInfo()
    {
        return HasTracking ? 
            $"Трек номер: STD-{DeliveryOptionId}-{DateTime.Now:yyyyMMdd}" : 
            "Отслеживание недоступно";
    }

    public void UpdatePackageInfo(string packageType, bool hasTracking, decimal insurancePercent)
    {
        PackageType = packageType;
        HasTracking = hasTracking;
        InsuranceCost = Cost * (insurancePercent / 100);
        
        Console.WriteLine($"Информация о посылке обновлена: Тип: {PackageType}, Отслеживание: {HasTracking}, Страховка: {InsuranceCost} руб.");
    }

    public override string GetProviderInfo()
    {
        return base.GetProviderInfo() + $", Тип упаковки: {PackageType}, Отслеживание: {HasTracking}";
    }

    public decimal CalculateInsuranceCost(decimal declaredValue)
    {
        if (declaredValue > 0)
        {
            return declaredValue * 0.02m;
        }
        return InsuranceCost;
    }

    public bool RequiresSpecialHandling()
    {
        return PackageType == "Хрупкая" || MaxWeight > 30;
    }

    public override string EstimateDeliveryTime() => 
        $"Среднее время: {AverageDeliveryTime} дней";

    public override string GetDeliveryDetails() => 
        base.GetDeliveryDetails() + $", Тип: Стандартная, {EstimateDeliveryTime()}, Макс. вес: {MaxWeight}кг, Регион: {DeliveryRegion}";

    public override string GetDeliveryDetails(bool includeStatus)
    {
        var details = GetDeliveryDetails();
        if (includeStatus)
        {
            details += $", Статус: {GetStatus()}, Регион доставки: {DeliveryRegion}";
        }
        return details;
    }

    public bool CanDeliverTo(string region)
    {
        return DeliveryRegion == "Все регионы" || DeliveryRegion == region;
    }

    public bool CanDeliverTo(string region, int weight)
    {
        return CanDeliverTo(region) && weight <= MaxWeight;
    }

    public decimal CalculateWeightSurcharge(int weight)
    {
        if (weight > MaxWeight)
            return Cost * 0.1m * (weight - MaxWeight);
        return 0;
    }

    public decimal CalculateWeightSurcharge(int weight, decimal multiplier)
    {
        if (weight > MaxWeight)
            return Cost * multiplier * (weight - MaxWeight);
        return 0;
    }

    public override bool CanCombineWith(ShippingOption other) => 
        other is Pickup;
}

public class ExpressDelivery : ShippingOption
{
    private int _minDeliveryTime;
    private decimal _expressFee;
    private bool _guaranteedDelivery;

    public int MinDeliveryTime 
    { 
        get => _minDeliveryTime;
        set => _minDeliveryTime = value > 0 ? value : throw new ArgumentException("Время доставки должно быть положительным");
    }

    public decimal ExpressFee 
    { 
        get => _expressFee;
        set => _expressFee = value >= 0 ? value : 0;
    }

    public bool GuaranteedDelivery 
    { 
        get => _guaranteedDelivery;
        set => _guaranteedDelivery = value;
    }

    public ExpressDelivery(string id, string name, decimal cost, int minTime, decimal expressFee = 100, bool guaranteed = true) 
        : base(id, name, cost)
    {
        MinDeliveryTime = minTime;
        ExpressFee = expressFee;
        GuaranteedDelivery = guaranteed;
    }

    public override decimal CalculateCost() => Cost * 1.5m + ExpressFee;

    public override string EstimateDeliveryTime() => 
        $"Минимальное время: {MinDeliveryTime} дней, Гарантия: {GuaranteedDelivery}";

    public override string GetDeliveryDetails() => 
        base.GetDeliveryDetails() + $", Тип: Экспресс, {EstimateDeliveryTime()}, Итог: {CalculateCost()} руб.";

    public override void UpdateCost(decimal newCost)
    {
        base.UpdateCost(newCost);
        Console.WriteLine($"Внимание: Экспресс-доставка имеет дополнительную плату {ExpressFee} руб.");
    }

    public void ScheduleDelivery(DateTime deliveryDate)
    {
        Console.WriteLine($"Экспресс-доставка запланирована на {deliveryDate:dd.MM.yyyy}");
    }

    public void ScheduleDelivery(DateTime deliveryDate, string timeSlot)
    {
        Console.WriteLine($"Экспресс-доставка запланирована на {deliveryDate:dd.MM.yyyy} в интервал {timeSlot}");
    }

    public bool IsUrgentDelivery()
    {
        return MinDeliveryTime <= 1;
    }

    public bool IsUrgentDelivery(int customUrgentThreshold)
    {
        return MinDeliveryTime <= customUrgentThreshold;
    }

    public override bool CanCombineWith(ShippingOption other) => 
        !(other is ExpressDelivery);
}

public class Pickup : ShippingOption
{
    private string _pickupAddress;
    private int _storageDays;
    private TimeSpan _workingHoursStart;

    public string PickupAddress 
    { 
        get => _pickupAddress;
        set => _pickupAddress = !string.IsNullOrEmpty(value) ? value : throw new ArgumentException("Адрес не может быть пустым");
    }

    public int StorageDays 
    { 
        get => _storageDays;
        set => _storageDays = value >= 1 ? value : 7;
    }

    public TimeSpan WorkingHoursStart 
    { 
        get => _workingHoursStart;
        set => _workingHoursStart = value;
    }

    public TimeSpan WorkingHoursEnd { get; set; }

    public Pickup(string id, string name, string address, int storageDays = 7) 
        : base(id, name, 0)
    {
        PickupAddress = address;
        StorageDays = storageDays;
        WorkingHoursStart = TimeSpan.FromHours(9);
        WorkingHoursEnd = TimeSpan.FromHours(21);
    }

    public override string GetDeliveryDetails() => 
        base.GetDeliveryDetails() + $", Тип: Самовывоз, Адрес: {PickupAddress}, Хранение: {StorageDays} дней";

    public override string EstimateDeliveryTime() => 
        $"Готов к выдаче через 2 часа, Время работы: {WorkingHoursStart:hh\\:mm}-{WorkingHoursEnd:hh\\:mm}";

    public override decimal CalculateCost() => 0;

    public override string GetDeliveryDetails(bool includeStatus, bool includePriority)
    {
        var details = GetDeliveryDetails();
        if (includeStatus)
        {
            details += $", Статус: {GetStatus()}";
        }
        if (includePriority)
        {
            details += $", Приоритет: {PriorityLevel}, Адрес: {PickupAddress}";
        }
        return details;
    }

    public bool IsWorkingNow()
    {
        var now = DateTime.Now.TimeOfDay;
        return now >= WorkingHoursStart && now <= WorkingHoursEnd;
    }

    public bool IsWorkingNow(DateTime customTime)
    {
        var time = customTime.TimeOfDay;
        return time >= WorkingHoursStart && time <= WorkingHoursEnd;
    }

    public DateTime GetExpiryDate()
    {
        return DateTime.Now.AddDays(StorageDays);
    }

    public DateTime GetExpiryDate(DateTime fromDate)
    {
        return fromDate.AddDays(StorageDays);
    }

    public override bool CanCombineWith(ShippingOption other) => 
        other is StandardDelivery;
}

public class DeliveryCollection<T> where T : ShippingOption
{
    private List<T> _items;

    public DeliveryCollection()
    {
        _items = new List<T>();
    }

    public void AddItem(T item)
    {
        _items.Add(item);
        Console.WriteLine($"Добавлено в коллекцию: {item.DeliveryOptionName}");
    }

    public void RemoveItem(string id)
    {
        var item = _items.Find(x => x.DeliveryOptionId == id);
        if (item != null)
        {
            _items.Remove(item);
            Console.WriteLine($"Удалено из коллекции: {item.DeliveryOptionName}");
        }
    }

    public T FindItem(string id)
    {
        return _items.Find(x => x.DeliveryOptionId == id);
    }

    public void DisplayAll()
    {
        Console.WriteLine($"\n=== Коллекция {typeof(T).Name} ===");
        foreach (var item in _items)
        {
            Console.WriteLine(item.GetDeliveryDetails());
        }
    }

    public void DisplayByStatus(bool isActive)
    {
        Console.WriteLine($"\n=== Опции со статусом {(isActive ? "Активна" : "Неактивна")} ===");
        foreach (var item in _items)
        {
            if (item.IsActive == isActive)
            {
                Console.WriteLine(item.GetDeliveryDetails());
            }
        }
    }

    public void DisplayByCost(decimal minCost, decimal maxCost)
    {
        Console.WriteLine($"\n=== Опции со стоимостью от {minCost} до {maxCost} руб. ===");
        foreach (var item in _items)
        {
            if (item.Cost >= minCost && item.Cost <= maxCost)
            {
                Console.WriteLine(item.GetDeliveryDetails());
            }
        }
    }
}

// Основной код программы (без класса Program)
Console.WriteLine("=== ДЕМОНСТРАЦИЯ НОВЫХ ВОЗМОЖНОСТЕЙ ===\n");

var container = new DependencyContainer();

var emailService = new EmailNotificationService("smtp.myserver.com", 587);
var discountService = new SeasonalDiscountService(0.15m);

container.Register<INotificationService>(emailService);
container.Register<IDiscountService>(discountService);

Console.WriteLine("=== УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ ===");
Console.WriteLine($"Email сервис зарегистрирован: {container.IsRegistered<INotificationService>()}");
Console.WriteLine($"Сервис скидок зарегистрирован: {container.IsRegistered<IDiscountService>()}");

var standard = new StandardDelivery("1", "Стандарт+", 300, 5, 
    container.GetService<INotificationService>(),
    container.GetService<IDiscountService>(),
    50, "Центральный регион");

Console.WriteLine("\n=== ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСОВ ===");

INotificationService notification = container.GetService<INotificationService>();
notification.SendNotification("Тестовое уведомление");
notification.SendNotification("Тестовое уведомление", "client@example.com");

IDiscountService discount = container.GetService<IDiscountService>();
decimal discountAmount = discount.CalculateDiscount(1000);
Console.WriteLine($"Скидка на 1000 руб.: {discountAmount} руб.");

Console.WriteLine("\n=== НОВЫЕ АТРИБУТЫ И МЕТОДЫ ===");

standard.SendCreationNotification();
Console.WriteLine($"Итоговая стоимость: {standard.CalculateFinalCost()} руб.");
Console.WriteLine($"Информация о провайдере: {standard.GetProviderInfo()}");

standard.UpdateProvider("Express Logistics", 4, true);

Console.WriteLine($"Информация об отслеживании: {standard.GetTrackingInfo()}");
standard.UpdatePackageInfo("Хрупкая", true, 2.5m);
Console.WriteLine($"Требуется специальная обработка: {standard.RequiresSpecialHandling()}");

Console.WriteLine("\n=== ПРОВЕРКА ВАЛИДАЦИИ ===");
Console.WriteLine($"Конфигурация валидна: {standard.ValidateConfiguration()}");
Console.WriteLine($"Конфигурация email сервиса: {emailService.GetConfiguration()}");
Console.WriteLine($"Email сервис валиден: {emailService.ValidateConfiguration()}");

Console.WriteLine("\n=== РАБОТА С СЕЗОННЫМИ СКИДКАМИ ===");
var seasonalService = (SeasonalDiscountService)discountService;
Console.WriteLine($"Сезон активен: {seasonalService.IsSeasonActive()}");
Console.WriteLine($"До конца сезона: {seasonalService.DaysUntilSeasonEnd()} дней");

decimal discountWithPromo = discount.CalculateDiscount(1000, "SUMMER2024");
Console.WriteLine($"Скидка с промокодом на 1000 руб.: {discountWithPromo} руб.");

Console.WriteLine("\n=== ИНФОРМАЦИЯ О БЕЗОПАСНОСТИ ===");
Console.WriteLine(standard.GetSecurityInfo());

=== ДЕМОНСТРАЦИЯ НОВЫХ ВОЗМОЖНОСТЕЙ ===

=== УПРАВЛЕНИЕ ЗАВИСИМОСТЯМИ ===
Email сервис зарегистрирован: True
Сервис скидок зарегистрирован: True

=== ЯВНАЯ РЕАЛИЗАЦИЯ ИНТЕРФЕЙСОВ ===
[EMAIL] Отправка уведомления: Тестовое уведомление
Через сервер: smtp.myserver.com:587
[EMAIL] Отправка уведомления для client@example.com: Тестовое уведомление
Через сервер: smtp.myserver.com:587
Применена сезонная скидка: 15% = 150.00 руб.
Скидка на 1000 руб.: 150.00 руб.

=== НОВЫЕ АТРИБУТЫ И МЕТОДЫ ===
[EMAIL] Отправка уведомления: Создана новая опция доставки: Стандарт+
Через сервер: smtp.myserver.com:587
Применена сезонная скидка: 15% = 45.00 руб.
Итоговая стоимость: 300 - 45.00 + 3.00 (страховка) = 258.00 руб.
Итоговая стоимость: 258.00 руб.
Информация о провайдере: Провайдер: Standard Provider, Макс. попыток доставки: 3, Требуется подпись: False, Тип упаковки: Стандартная, Отслеживание: True
Информация о провайдере обновлена: Провайдер: Express Logistics, Макс. попыток доставки: 4, Требуется подпис